# How far the optimal route moves when the objective changes

The three objectives of PASSAGE disagree about where to go. This notebook measures that disagreement at
every resolution: for the 64 validation preview scenes of each configuration, it compares the optimal
routes of each pair of objectives, on free terrain and with the obstacles blocked. A planner that learns
one objective cannot simply reuse its routes for another.

The distance is the 95th-percentile Hausdorff distance (H95): the distance, in cells, that 95 % of both
routes stay within. It is reported in percent of the grid side so that resolutions can be compared.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It decodes 64 preview scenes per resolution; a few minutes in total.

In [ ]:
resolutions = [64, 128, 256, 512, 1024, 2048, 4096]
split = "validation"
scenes = 64  # preview scenes per resolution
board_resolution = 256  # resolution of the scene drawn in Section 1
output_dir = "notebook-artifacts/notebooks/paper/cost_model_path_divergence"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "2ab44d899652c7fb99c425488e84a7d565963547"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. One scene, six routes

Figure 1 shows the six stored routes of one validation scene at the board resolution, in the paper's
style: columns are objectives, rows are conditions. The scene is the first whose routes all span at
least a quarter of the grid side.

In [ ]:
row = source.get_row(board_resolution, split, sample_id=first_scene(board_resolution, split), samples=False)
board = render_cost_comparison(row, title=f"Scene {row['id']} · {board_resolution}×{board_resolution} · {split}")
figures = save_figure(board, output_path, f"cost_model_path_divergence-{board_resolution}-{row['id']}")
plt.show()

*Figure 1. The six stored routes of one scene. Green solid: free terrain; blue dashed: obstacles blocked.*

## 2. Divergence at every resolution

In [ ]:
from passage.notebook_reports import route_metrics_report

NAMES = {"astar_elevation": "Elevation", "astar_slope": "Slope (uphill)", "astar_energy": "Energy"}
PAIR_COLORS = {"Elevation vs Slope (uphill)": "#1f8a70", "Elevation vs Energy": "#b07d1a",
               "Energy vs Slope (uphill)": "#3a62b5"}

def objective_of(route):
    return NAMES["_".join(route.split("_")[1:3])]

rows = []
for resolution in resolutions:
    for preview in source.iter_rows(resolution, split, columns=None, samples=True, limit=scenes):
        for c in route_metrics_report(preview, resolution, split)["comparisons"]:
            a, b = objective_of(c["route"]), objective_of(c["reference_route"])
            if a == b or c["hausdorff_p95_pixels"] is None:
                continue
            rows.append({"resolution": resolution, "pair": " vs ".join(sorted((a, b))),
                         "condition": c["route"].rsplit("_", 1)[1], "assd": c["assd"],
                         "h95_share": 100 * c["hausdorff_p95_pixels"] / resolution})
divergence = pd.DataFrame(rows)
table = (divergence[divergence.condition == "free"].groupby(["pair", "resolution"]).h95_share.median()
         .unstack("resolution").round(2))
table.columns = [f"{r}×{r}" for r in table.columns]
show(table.reset_index(names="Objective pair"),
     "Table 1. Median H95 between the optima of two objectives, free terrain (% of grid side).", "{:,.2f}")
identical = (divergence.groupby(["pair", "condition"]).assd.apply(lambda s: 100 * (s == 0).mean())
             .unstack("condition").round(1))
show(identical.reset_index(names="Objective pair").rename(columns={"free": "Free terrain (%)", "obstacles": "With obstacles (%)"}),
     "Table 2. Share of scenes where two objectives give the very same route, all resolutions.", "{:,.1f}")

fig, ax = plt.subplots(figsize=(8.6, 3.8))
fig.subplots_adjust(top=0.8)
for pair, color in PAIR_COLORS.items():
    for condition, style in (("free", "-"), ("obstacles", (0, (3.2, 1.7)))):
        med = divergence[(divergence.pair == pair) & (divergence.condition == condition)].groupby("resolution").h95_share.median()
        ax.plot(med.index, med.values, color=color, linestyle=style, marker="o", markersize=4.5,
                label=f"{pair} · {'free' if condition == 'free' else 'obstacles'}")
ax.set_xscale("log", base=2)
ax.set_xticks(resolutions, [str(r) for r in resolutions])
ax.set(xlabel="Grid side (cells)", ylabel="Median H95 (% of grid side)")
ax.legend(fontsize=7.5, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.2))
fig.subplots_adjust(bottom=0.3)
figure_title(fig, "Distance between the optima of two objectives, per resolution",
             f"{scenes} {split} preview scenes per resolution · solid: free terrain · dashed: obstacles blocked")
figures += save_figure(fig, output_path, "cost_model_path_divergence-resolutions")
plt.show()

*Figure 2. Median distance between the optimal routes of two objectives, in percent of the grid side,
against the resolution. Solid lines: free terrain. Dashed lines: obstacles blocked.*

## Summary

In [ ]:
free = divergence[divergence.condition == "free"]
by_pair = free.groupby("pair").h95_share.median().sort_values()
same = divergence.groupby("pair").assd.apply(lambda s: 100 * (s == 0).mean())
display(Markdown("\n".join([
    f"- Over {divergence.resolution.nunique()} resolutions and {scenes} scenes each, the median H95 on free "
    "terrain is " + ", ".join(f"{v:.1f} % of the side ({k.lower()})" for k, v in by_pair.items()) + ".",
    "- The two objectives give the very same route in " + ", ".join(
        f"{v:.0f} % of the comparisons ({k.lower()})" for k, v in same.items()) + ".",
    f"- For the most distant pair, {by_pair.index[-1].lower()}, the median H95 on free terrain ranges from "
    f"{free[free.pair == by_pair.index[-1]].groupby('resolution').h95_share.median().min():.1f} to "
    f"{free[free.pair == by_pair.index[-1]].groupby('resolution').h95_share.median().max():.1f} % of the side "
    "across resolutions; for the closest pair it stays at "
    f"{free[free.pair == by_pair.index[0]].groupby('resolution').h95_share.median().max():.1f} % or less.",
])))
report = {"resolutions": resolutions, "scenes": scenes, "median_h95_share_free": table.to_dict(),
          "identical_route_share": identical.to_dict(), "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, "cost_model_path_divergence.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*